# Sec 5.1 / Figure 3 -- CNTK-Nystrom + Logistic Regression (CIFAR-10)

Notebook riêng, tách ra từ `algorithm1_cntk_experiments.ipynb` (vốn chạy chung CIFAR-10 +
FashionMNIST), **chỉ chạy CIFAR-10** — đúng thực nghiệm gốc của paper Mục 5.1 (bilevel
coreset cho hồi quy logistic đa lớp trên không gian đặc trưng Nyström của CNTK 6 lớp +
global average pooling).

**HƯỚNG DẪN:**
1. Bật GPU (T4 x2 hoặc P100) trong Settings của Kaggle (hoặc Runtime type trên Colab).
2. Chạy Ô Số 1 để cài đặt môi trường + tải mã nguồn.
3. Bấm **Restart Session/Runtime** khi được yêu cầu.
4. Chạy tiếp Ô vá JAX + Ô kiểm tra GPU + Ô cấu hình.
5. Chạy Ô smoke test trước (tham số nhỏ, chỉ kiểm tra pipeline chạy được).
6. Chạy Ô đầy đủ (đúng tham số paper) -- nặng, tính CNTK-Nystrom feature cho toàn bộ
   CIFAR-10 có thể mất hàng giờ, nhưng chỉ tính 1 lần rồi cache lại.


### Ô Số 0: Kiểm tra GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "KHONG CO GPU! Vao Settings > Accelerator (Kaggle) hoac Runtime > Change runtime "
        "type (Colab), chon GPU, roi chay lai notebook tu dau."
    )
print('GPU OK:', torch.cuda.get_device_name(0))


In [ ]:
# Ô SỐ 1: CÀI ĐẶT MÔI TRƯỜNG VÀ TẢI MÃ NGUỒN
!git clone https://github.com/quachthanhhmd/bilevel-coresets.git
%cd bilevel-coresets
!git checkout master


⚠️ **CẢNH BÁO QUAN TRỌNG:** Dừng lại tại đây! Bạn phải bấm `Restart Session` (hoặc `Restart Kernel`/`Restart Runtime`) trước khi chạy ô tiếp theo.

In [ ]:
# 1. Hạ cấp setuptools để vá lỗi môi trường build của Python 3.12
!pip install "setuptools<70.0.0"

# 2. Cài duy nhất thư viện mô phỏng mạng CNN còn thiếu
!pip install neural-tangents

!pip install --upgrade jax jaxlib==0.1.56+cuda101 -f https://storage.googleapis.com/jax-releases/jax_releases.html


In [ ]:
import jax
import jax.core
import jax._src.core
import jax.tree_util
import jax.util

missing_classes = [
    'Jaxpr', 'JaxprEqn', 'Literal', 'Var', 'DropVar',
    'ClosedJaxpr', 'ShapedArray', 'Value', 'MainTrace', 'Trace', 'Primitive'
]
for cls_name in missing_classes:
    if hasattr(jax._src.core, cls_name):
        setattr(jax.core, cls_name, getattr(jax._src.core, cls_name))

if not hasattr(jax.tree_util, 'tree_multimap'):
    jax.tree_util.tree_multimap = jax.tree_util.tree_map

def custom_safe_map(f, *args):
    return list(map(f, *args))

def custom_safe_zip(*args):
    return list(zip(*args))

jax.util.safe_map = custom_safe_map
jax.util.safe_zip = custom_safe_zip

print("Đã vá nóng JAX. Sẵn sàng nạp Neural Tangents.")


### Ô Số 2: Cấu hình

In [ ]:
import os, subprocess

# Tự dò thư mục repo đã clone -- chạy được trên cả Kaggle lẫn Colab.
_candidates = [
    '/kaggle/working/bilevel-coresets',
    '/content/bilevel-coresets',
    os.path.join(os.getcwd(), 'bilevel-coresets'),
]
REPO = next((c for c in _candidates if os.path.isdir(c)), None)
if REPO is None:
    raise RuntimeError(
        "Khong tim thay thu muc 'bilevel-coresets'. Kiem tra lai da chay xong O SO 1 chua."
    )
os.chdir(REPO)
print('REPO =', REPO)


## Ô Số 3: Smoke test (khuyến nghị chạy trước)

Dùng tham số nhỏ (`--nystrom-dim`, `--train-pool-size`...) -- KHÔNG phải mặc định của
script -- chỉ để kiểm tra jax/neural-tangents, tải CIFAR-10, và toàn bộ pipeline chạy
được trên môi trường hiện tại trước khi cam kết chạy bản đầy đủ (có thể mất hàng giờ).


In [ ]:
smoke_common = [
    '--dataset', 'cifar10',
    '--nystrom-dim', '128', '--train-pool-size', '2000', '--val-size', '200',
    '--first-inner-it', '200', '--max-inner-it', '50', '--cg-iters', '10',
    '--features-cache-dir', 'experiments/cifar10_cntk_features_smoketest',
    '--output-dir', 'experiments/algo1_paper_cifar10_results_smoketest',
]

subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'full', '--seed', '0'] + smoke_common, check=True)
subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'bico_fwd', '--size-pct', '10', '--seed', '0'] + smoke_common, check=True)
subprocess.run(['python', 'experiments/run_algorithm1_variants_paper_cifar10.py',
                '--method', 'bico_reg', '--size-pct', '10', '--seed', '0',
                '--reg-outer-it', '10'] + smoke_common, check=True)
print("Smoke test OK -- có thể chạy Ô Số 4 bên dưới.")


## Ô Số 4: Chạy đầy đủ (đúng tham số paper, chỉ CIFAR-10)

Gọi `run_algorithm1_paper_experiments.sh --datasets cifar10` -- đúng script gốc của
`algorithm1_cntk_experiments.ipynb`, chỉ giới hạn lại 1 dataset thay vì chạy cả 2. Đúng
tham số Appendix C mặc định (không tự scale nhỏ). Nặng -- chỉ chạy sau khi Ô Số 3 đã OK.

Kết quả (để backup/tải về): `experiments/algo1_paper_cifar10_results/`. Chạy lại với
`--stage report` sau này để chỉ vẽ lại biểu đồ mà không cần build lại feature CNTK-Nystrom
(rất tốn thời gian).


In [ ]:
!chmod +x run_algorithm1_paper_experiments.sh
!./run_algorithm1_paper_experiments.sh --datasets cifar10


## Ô Số 5: Xem biểu đồ kết quả

In [ ]:
from IPython.display import Image, display
display(Image(filename='experiments/algorithm1_variants_paper_cifar10_accuracy.png'))
